# Day 19 · Senses: Smell — แผนที่กลิ่นของไทยจากก๊าซห้าชนิดกับทิศลม

**#30DayMapChallenge 2026 · 19 พ.ย. 2569** · โจทย์:
*"How do you map things which your nose can sense? What does a smelly map look like?"*
(เขียนคืนวันที่ 19 ต่อเช้าวันที่ 20 — ช้าไม่กี่ชั่วโมง บอกไว้ในโพสต์)

Overpass ล่มทั้งคืน (502) จึงไม่ใช้แหล่งกำเนิดกลิ่นจาก OSM แต่ใช้ **ก๊าซที่พาโมเลกุลไปถึงจมูก**
จากแบบจำลอง CAMS (ผ่าน Open-Meteo Air Quality API) บวก **ลม** จาก ERA5
ที่ 83 จุดทะเบียน ช่วง 16–19 พ.ย. 2569 (96 ชั่วโมง)

| ตัววัด | นิยาม |
|---|---|
| CH4 / SO2 / CO / NO2 / PM2.5 | ค่าเฉลี่ยรายชั่วโมง (µg/m³) ที่ grid cell ของ CAMS ครอบคลุมจุดนั้น |
| z-score | (ค่าจุด − ค่าเฉลี่ย 83 จุด) ÷ SD ของ 83 จุด — ทำให้ก๊าซคนละหน่วยบวกกันได้ |
| smell_index | ค่าเฉลี่ย z ของห้าก๊าซ (ตัวจัดอันดับ ไม่ใช่หน่วยกลิ่น ou/m³) |
| signature | ก๊าซที่มี z สูงสุด (ไม่ใช่ µg/m³ สูงสุด — ดูบทเรียน) |
| wind_from_dir | ทิศที่ลมพัด *มา* เฉลี่ยแบบเวกเตอร์ u/v ถ่วงด้วยความเร็ว |

**รัน 2 ไฟล์ตามลำดับ**
1. `tools/collect_day19_smell.py` → `cache/day19/site_NN.json` (มีแคช ข้ามจุดที่ทำแล้ว)
2. `tools/day19_smell.py` → ภาพ 3 แผ่น + csv 2 ไฟล์ + `day19_stats.json`

**สิ่งที่ไม่มีอยู่จริงและต้องพูดดัง ๆ**: ammonia (NH3) — กลิ่นหลักของนาข้าว คอกสัตว์ บ่อขยะ —
CAMS ระดับโลกคืน null ทั้ง 96 ชั่วโมงทุกจุดในไทย, dust = 0 ทุกจุด,
และจุดหลายคู่ถูก map ไป grid cell เดียวกันเพราะ resolution ~0.1°
ทั้งหมดบันทึกใน `unavailable` และ `no_data_variables` ของ stats

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

## ขั้นที่ 0 — ดึงข้อมูลอากาศกับลมก่อน (ต้องรันบนเครื่องที่มีเน็ต)

เซลล์ถัดไปคือ `tools/collect_day19_smell.py` ต้นทาง ให้ได้ออกมาเป็น `cache/day19/site_NN.json` ครบ 83 จุด (มีแคช ข้ามจุดที่ทำแล้ว)

In [ ]:
# -*- coding: utf-8 -*-
"""
collect_day19_smell.py — ดึงข้อมูล "สิ่งที่จมูกจับได้" ให้ 83 จุดทะเบียน

เหตุผลที่เลือกชั้นข้อมูลนี้: Overpass คิวแน่นอีกแล้วคืนนี้ (502) จึงสร้างแผ่นจาก
แหล่งที่เชื่อถือได้และตอบคำถาม "กลิ่น" ได้ตรงกว่าที่คิด — ความเข้มข้นของก๊าซที่
พากลิ่นมาจริง (methane จากนาข้าว/ขยะ/ปศุสัตว์, SO2 จากอุตสาหกรรม, CO/PM2.5 จากการ
เผาไร่อ้อยซึ่งคือกลิ่นของฤดูพฤศจิกายนในไทย) จาก CAMS ผ่าน Open-Meteo air-quality API
บวกทิศทางลมเฉลี่ยแบบเวกเตอร์จาก ERA5 ผ่าน Open-Meteo archive API

หมายเหตุความซื่อสัตย์: ammonia (NH3) ที่ควรเป็นตัวชี้กลิ่นปศุสัตว์/นาข้าว
*ไม่มีข้อมูล* ในโมเดลระดับโลกสำหรับไทย (API คืน null ทั้ง 192 ชั่วโมง) — บันทึกไว้
ใน stats ว่าลองแล้วไม่มี ไม่ใช่ลืม
"""
import os, sys, json, time, math, urllib.request
import pandas as pd

WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day19")
os.makedirs(CACHE, exist_ok=True)
UA = "geo-notes-day19/1.0 (30DayMapChallenge research)"
AQ_VARS = "ammonia,methane,sulphur_dioxide,carbon_monoxide,nitrogen_dioxide,pm2_5,pm10,ozone,dust"
START, END = "2026-11-16", "2026-11-19"


def get(url, tries=4):
    for a in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=90) as r:
                return json.loads(r.read().decode("utf-8"))
        except Exception as e:
            print(f"   retry {a+1}: {type(e).__name__} {str(e)[:60]}", flush=True)
            time.sleep(3 + 3 * a)
    return None


df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"), dtype=str, keep_default_na=False)
df["name"] = df["name"].astype(str).str.strip()
out = {}
for i, r in df.iterrows():
    lat, lon = float(r.latitude), float(r.longitude)
    path = os.path.join(CACHE, f"site_{i:02d}.json")
    if os.path.exists(path) and os.path.getsize(path) > 200:
        out[str(i)] = json.load(open(path, encoding="utf-8"))
        continue
    aq = get("https://air-quality-api.open-meteo.com/v1/air-quality"
             f"?latitude={lat:.4f}&longitude={lon:.4f}&hourly={AQ_VARS}"
             f"&start_date={START}&end_date={END}&timezone=Asia%2FBangkok")
    wd = get("https://archive-api.open-meteo.com/v1/archive"
             f"?latitude={lat:.4f}&longitude={lon:.4f}&start_date={START}&end_date={END}"
             "&hourly=wind_direction_10m,wind_speed_10m&timezone=Asia%2FBangkok")
    if aq is None or wd is None:
        print(f"[{i:02d}] พัง — จะรันใหม่เฉพาะจุดที่ขาด", flush=True)
        continue
    rec = dict(idx=int(i), name=r["name"], lat=lat, lon=lon,
               aq_time=aq["hourly"]["time"], aq=aq["hourly"], aq_units=aq.get("hourly_units"),
               wind_time=wd["hourly"]["time"], wind=wd["hourly"],
               aq_grid=(aq.get("latitude"), aq.get("longitude")),
               wind_grid=(wd.get("latitude"), wd.get("longitude")),
               fetched_utc=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
    json.dump(rec, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    out[str(i)] = rec
    if i % 10 == 0:
        print(f"[{i:02d}] {r['name'][:24]:24} ok", flush=True)
    time.sleep(0.25)

have = len(out)
print(f"COLLECT_DONE ได้ {have} / {len(df)} จุด")
if have < len(df):
    print("จุดที่ขาด:", [i for i in range(len(df)) if str(i) not in out][:20])

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 19 · Senses: Smell — "How do you map things which your nose can sense?
What does a smelly map look like?"

ข้อความโจทย์ดึงจาก 30daymapchallenge.com อ่านสดวันที่ 20 พ.ย.

คำตอบของแผ่นนี้: "แผนที่กลิ่น" ที่วาดจาก *ก๊าซที่พากลิ่นมา* ไม่ใช่จากความรู้สึก
  methane  = การย่อยสลายอินทรีย์ นาข้าว_b่อขยะ_ปศุสัตว์
  SO2      = อุตสาหกรรม / การเผาเชื้อเพลิงกำมะถัน
  CO       = การเผาชีวมวล (ซึ่งคือกลิ่นของฤดูกุมภาพันธ์–เมษายน และปลายพฤศจิกายนในไทย)
  NO2      = การจราจร/เผาไหม้อุณหภูมิสูง
  PM2.5    = ควัน (พาอนุภาคที่จมูกรับได้)
บวก *ลม* เพราะกลิ่นเดินทางตามลม แผนที่กลิ่นที่ไม่บอกทิศทางลมคือแผนที่ที่โกหก

ข้อจำกัดที่รู้ก่อนวาด (บันทึกไว้ใน stats ไม่ใช่ซ่อน):
  • ammonia (NH3) ซึ่งควรเป็นตัวชี้กลิ่นปศุสัตว์/นาข้าว API คืน null ทั้ง 192 ชั่วโมง
    สำหรับทุกจุดในไทย — โมเดลระดับโลก (CAMS) ไม่มีตัวนี้ให้ region นี้
  • dust = 0 ทุกชั่วโมงทุกจุดในช่วงที่วัด
  • ค่าทั้งหมดเป็น *แบบจำลอง* บน grid ~0.1–0.25° ไม่ใช่การวัดภาคสนาม และไม่ใช่หน่วย ou/m³
  • หน้าต่างเวลา 16–19 พ.ย. 2569 เป็น snapshot สี่วัน ผลจะต่างถ้าเปลี่ยนสัปดาห์

ทุกตัวเลขอ่านจาก day19_smell_sites.csv / day19_by_region.csv / day19_stats.json เท่านั้น
"""
import os, sys, json, time, math, glob, collections
import numpy as np, pandas as pd, geopandas as gpd
from shapely.validation import make_valid
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection
from scipy.stats import spearmanr

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06")
SM = os.path.join(WORK, "cache", "day19")
OUT = os.path.join(WORK, "out", "day19")
os.makedirs(OUT, exist_ok=True)
for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day19_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

sys.path.insert(0, os.path.join(WORK, "tools"))
from guards import guard_bbox, guard_crs, guard_variance, lint_thai

TXT = []
def T(s):
    TXT.append(s); return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK, MUTED = "#0d1522", "#e8e3d3", "#9fb2c8"
ACCENT, WARM, COOL, TEAL = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf"
GASES = ["methane", "sulphur_dioxide", "carbon_monoxide", "nitrogen_dioxide", "pm2_5"]
WINDOW_START, WINDOW_END = "2026-11-16", "2026-11-19"
LABEL = {"methane": "CH4", "sulphur_dioxide": "SO2", "carbon_monoxide": "CO",
         "nitrogen_dioxide": "NO2", "pm2_5": "PM2.5", "ammonia": "NH3", "dust": "dust",
         "pm10": "PM10", "ozone": "O3"}
MARK = {"methane": "o", "sulphur_dioxide": "s", "carbon_monoxide": "^",
        "nitrogen_dioxide": "D", "pm2_5": "p"}

## ตอน 1) ข้อมูล

In [ ]:
# ---------------------------------------------------------------- 1. ข้อมูล
files = sorted(glob.glob(os.path.join(SM, "site_*.json")))
if len(files) != 83:
    raise SystemExit(f"ข้อมูลไม่ครบ: มี {len(files)} / 83 จุด — รัน tools/collect_day19_smell.py ต่อ")

df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"), dtype=str, keep_default_na=False)
df["name"] = df["name"].astype(str).str.strip()
assert len(df) == 83
SITES = np.column_stack([df.longitude.values.astype(float), df.latitude.values.astype(float)])
guard_bbox(SITES, 96.5, 107.6, 4.3, 21.7, "darksky-sites")
ne = gpd.read_file(os.path.join(CACHE, "ne_10m_admin_0_countries.shp"))
TH = make_valid(ne[ne.NAME == "Thailand"].geometry.iloc[0])
PARTS = list(TH.geoms)
guard_crs(TH.bounds, "thailand-outline", expect="degrees")

def mean_of(vals):
    v = [x for x in vals if isinstance(x, (int, float))]
    return (float(np.mean(v)), len(v)) if v else (np.nan, 0)

rows, null_report = [], collections.Counter()
for f in files:
    rec = json.load(open(f, encoding="utf-8"))
    i = int(rec["idx"])
    d = dict(idx=i, name=rec["name"], lat=rec["lat"], lon=rec["lon"],
             region_th=str(df.region_th.iloc[i]).strip(),
             province=str(df.province.iloc[i]).strip(),
             sqm=float(df.sky_quality_sqm.iloc[i]) if str(df.sky_quality_sqm.iloc[i]).strip() else np.nan,
             grid_aq=rec.get("aq_grid"), grid_wind=rec.get("wind_grid"))
    for g in list(GASES) + ["ammonia", "dust", "pm10", "ozone"]:
        m, n = mean_of(rec["aq"].get(g, []))
        d[LABEL[g]] = m
        d[LABEL[g] + "_n"] = n
        if n == 0:
            null_report[g] += 1
    # ลม: ค่าเฉลี่ยแบบเวกเตอร์ (ทิศที่ลม *พัดมา* ตามธรรมเนียมอุตุนิยมวิทยา)
    u10 = rec["wind"].get("wind_direction_10m", [])
    s10 = rec["wind"].get("wind_speed_10m", [])
    xs, ys, ss, nn = 0.0, 0.0, 0.0, 0
    for a, v in zip(u10, s10):
        if isinstance(a, (int, float)) and isinstance(v, (int, float)):
            rad = math.radians(a)
            xs += v * math.sin(rad); ys += v * math.cos(rad); ss += v; nn += 1
    d["wind_from_dir"] = round(math.degrees(math.atan2(xs, ys)) % 360.0, 2) if nn else np.nan
    d["wind_mean_speed"] = round(ss / nn, 3) if nn else np.nan
    d["wind_n"] = nn
    rows.append(d)

S = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)
for c in GASES:
    assert S[LABEL[c]].notna().all(), f"{LABEL[c]} มีค่าว่าง"
guard_variance(S["CH4"].values, "ch4", min_spread_km=0)

## ตอน 2) z-score และดัชนีกลิ่น

In [ ]:
# ------------------------------------------------- 2. z-score และดัชนีกลิ่น
Z = {}
for c in GASES:
    v = S[LABEL[c]].values.astype(float)
    Z[LABEL[c]] = (v - v.mean()) / v.std(ddof=0)
    S[f"z_{LABEL[c]}"] = Z[LABEL[c]]
S["smell_index"] = np.round(np.mean([Z[LABEL[c]] for c in GASES], axis=0), 4)
S["signature"] = [LABEL[GASES[j]] for j in np.argmax(np.vstack([Z[LABEL[c]] for c in GASES]), axis=0)]
sig_counts = collections.Counter(S.signature.tolist())
log("ก๊าซที่เด่นของแต่ละจุด: " + " · ".join(f"{k} {v} จุด" for k, v in sig_counts.most_common()))
log(f"smell_index (ผลรวม z ของ 5 ก๊าซ): ต่ำสุด {S.smell_index.min():.3f} · "
    f"มัธยฐาน {S.smell_index.median():.3f} · สูงสุด {S.smell_index.max():.3f}")
log("ค่าเฉลี่ยต่อจุด (µg/m3): " + " · ".join(
    f"{LABEL[c]} {S[LABEL[c]].mean():,.1f}" for c in GASES))
log("ตัวแปรที่ไม่มีข้อมูลเลย: " + (", ".join(f"{LABEL[k]} ({v}/83 จุด)" for k, v in null_report.items())
                                  or "ไม่มี"))

## ตอน 3) เทียบกับซีรีส์ (ความมืด / ความเป็นเมือง)

In [ ]:
# ------------------------------------------------- 3. เทียบกับซีรีส์ (ความมืด / ความเป็นเมือง)
rho_sqm, p_sqm = spearmanr(S.smell_index.values, S.sqm.values)
d16 = pd.read_csv(os.path.join(REPO, "posts/day16-collaborative-map/day16_sites_osm.csv"))
ways = d16.ways.values
rho_ways, p_ways = spearmanr(S.smell_index.values, ways)
by_region = (S.assign(smell=S.smell_index).groupby("region_th")
             .agg(sites=("idx", "size"), smell_mean=("smell", "mean"),
                  smell_min=("smell", "min"), smell_max=("smell", "max"),
                  ch4=("CH4", "mean"), so2=("SO2", "mean"), co=("CO", "mean"),
                  no2=("NO2", "mean"), pm=("PM2.5", "mean"),
                  z_ch4=("z_CH4", "mean"), z_so2=("z_SO2", "mean"), z_co=("z_CO", "mean"),
                  z_no2=("z_NO2", "mean"), z_pm=("z_PM2.5", "mean"),
                  wind=("wind_mean_speed", "mean")).reset_index())
# "ก๊าซที่เด่น" ต้องเทียบด้วย z-score ไม่ใช่ µg/m³ดิบ
# เพราะ CH4 มีพื้นหลังระดับพัน µg/m³ ส่วน SO2 มีหลักหน่วย — เทียบดิบคือคนละหน่วยกัน
DOM = {"CH4": "z_ch4", "SO2": "z_so2", "CO": "z_co", "NO2": "z_no2", "PM2.5": "z_pm"}
by_region["dominant"] = [max(((by_region.loc[j, col], gas) for gas, col in DOM.items()))[1]
                         for j in range(len(by_region))]
log(f"กลิ่นกับฟ้า: Spearman(smell_index, SQM) = {rho_sqm:.3f} (p = {p_sqm:.2g})")
log(f"กลิ่นกับแผนที่: Spearman(smell_index, ways Day 16) = {rho_ways:.3f} (p = {p_ways:.2g})")
AGG = dict(sites=int(len(S)), window=f"{WINDOW_START} ถึง {WINDOW_END}",
           smell_min=round(float(S.smell_index.min()), 4),
           smell_median=round(float(S.smell_index.median()), 4),
           smell_mean=round(float(S.smell_index.mean()), 4),
           smell_max=round(float(S.smell_index.max()), 4),
           signature_counts={k: int(v) for k, v in sig_counts.items()},
           gas_mean={LABEL[c]: round(float(S[LABEL[c]].mean()), 3) for c in GASES},
           gas_min={LABEL[c]: round(float(S[LABEL[c]].min()), 3) for c in GASES},
           gas_max={LABEL[c]: round(float(S[LABEL[c]].max()), 3) for c in GASES},
           wind_speed_mean=round(float(S.wind_mean_speed.mean()), 3),
           wind_speed_min=round(float(S.wind_mean_speed.min()), 3),
           wind_speed_max=round(float(S.wind_mean_speed.max()), 3),
           spearman_smell_vs_sqm=round(float(rho_sqm), 3), p_smell_sqm=float(p_sqm),
           spearman_smell_vs_ways=round(float(rho_ways), 3), p_smell_ways=float(p_ways),
           no_data_variables={LABEL[k]: int(v) for k, v in null_report.items()})
START, END = "2026-11-16", "2026-11-19"
AGG["window"] = f"{START} ถึง {END}"

# ================================================== ภาพ 1 — แผนที่กลิ่น
def draw_hero():
    fig = plt.figure(figsize=(15.6, 9.0), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.02, 0.115, 0.665, 0.775]); ax.set_facecolor("#05080f")
    ax.add_collection(PolyCollection([list(p.exterior.coords) for p in PARTS],
                                     facecolor="#101a2c", edgecolor="#4a6288", lw=0.9))
    norm = plt.Normalize(AGG["smell_min"], AGG["smell_max"])
    cmap = plt.get_cmap("inferno")
    for _, r in S.iterrows():
        # เส้นลม: ลากจากจุดไปทิศที่ลมพัด *ไป* (ทิศอุตุนิยมวิทยา = พัดมาจาก wind_from_dir)
        to_rad = math.radians((r.wind_from_dir + 180.0) % 360.0)
        L = 0.16 + 0.62 * (r.wind_mean_speed / max(AGG["wind_speed_max"], 0.1))
        dx, dy = L * math.sin(to_rad), L * math.cos(to_rad)
        ax.plot([r.lon, r.lon + dx], [r.lat, r.lat + dy], color=cmap(norm(r.smell_index)),
                lw=1.5, alpha=0.85, zorder=4, solid_capstyle="round")
        ax.scatter([r.lon + dx], [r.lat + dy], marker=MARK.get(r.signature, "o"), s=78,
                   color=cmap(norm(r.smell_index)), edgecolor=INK, lw=0.4, zorder=6)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.add_axes([0.70, 0.42, 0.016, 0.40])
    plt.colorbar(sm, cax=cb).set_label(T("ดัชนีกลิ่น (ค่าเฉลี่ย z-score ของ 5 ก๊าซ)"),
                                       color=INK, fontsize=9.5)
    cb.tick_params(colors=INK, labelsize=8.5)
    txt = fig.add_axes([0.70, 0.115, 0.29, 0.28]); txt.axis("off")
    txt.text(0, 1.0, T("เครื่องหมายท้ายเส้นลม = ก๊าซที่เด่นที่จุดนั้น"), color=INK, fontsize=11)
    keys = [("o", "CH4 (อินทรีย์/นาข้าว/ขยะ)"), ("s", "SO2 (อุตสาหกรรม)"),
            ("^", "CO (เผาชีวมวล)"), ("D", "NO2 (จราจร/เผาไหม้)"),
            ("p", "PM2.5 (ควัน)")]
    y = 0.86
    for m, lab in keys:
        txt.scatter([0.02], [y], marker=m, s=48, color=ACCENT, transform=txt.transAxes)
        txt.text(0.07, y, T(lab), color=MUTED, fontsize=9.8, va="center", transform=txt.transAxes)
        y -= 0.17
    side = fig.add_axes([0.70, 0.015, 0.29, 0.11]); side.axis("off")
    side.text(0, 0.96, T(f"ลมเฉลี่ย {AGG['wind_speed_mean']:.2f} ม./วิ "
                         f"(ช่วง {AGG['wind_speed_min']:.2f}–{AGG['wind_speed_max']:.2f})"),
              color=MUTED, fontsize=9.8)
    side.text(0, 0.52, T("เส้นลมชี้ทิศที่ลม *พัดไป*" + chr(10) +
                         "กลิ่นไม่ได้อยู่เฉย ๆ มันเดินทาง" + chr(10) +
                         "แผนที่กลิ่นที่ไม่บอกทิศลมคือแผนที่ที่โกหก"),
              color=MUTED, fontsize=9.8, va="top")
    ax.set_xlim(97.0, 106.8); ax.set_ylim(5.0, 21.1); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(T(f"83 จุดทะเบียน · 16–19 พ.ย. 2569 · ค่าเฉลี่ยรายชั่วโมงของ CAMS บน grid ~0.1–0.25°"),
                 color=INK, fontsize=11.5, pad=8)
    fig.text(0.02, 0.055, T("ไม่มีข้อมูล NH3 (แอมโมเนีย) สำหรับประเทศไทยในโมเดลระดับโลก — "
                            "กลิ่นปศุสัตว์และนาข้าวจึงหายไปจากแผ่นนี้ทั้งที่ควรเป็นตัวหลัก"),
             color=WARM, fontsize=10.2)
    fig.suptitle(T("แผนที่กลิ่น · สิ่งที่จมูกจับได้ในคืนที่ฟ้ามืด"), color=INK, fontsize=16, y=0.985)
    p = os.path.join(OUT, "day19_smell.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ================================================== ภาพ 2 — ไทยกลิ่นอะไร ตามภูมิภาค
def draw_regions():
    order = by_region.sort_values("smell_mean", ascending=False)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15.2, 6.2), dpi=150,
                                  gridspec_kw=dict(width_ratios=[1.25, 1]))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    gases = ["CH4", "SO2", "CO", "NO2", "PM2.5"]
    cols = {"CH4": TEAL, "SO2": WARM, "CO": ACCENT, "NO2": COOL, "PM2.5": "#a78bfa"}
    # normalise each gas to its own max so five gases share one axis
    mx = {g: max(order[g].values) for g in ["ch4", "so2", "co", "no2", "pm"]}
    key = {"CH4": "ch4", "SO2": "so2", "CO": "co", "NO2": "no2", "PM2.5": "pm"}
    y = np.arange(len(order))
    for k, g in enumerate(gases):
        vals = order[key[g]].values / mx[key[g]] * 100
        ax.barh(y, vals, left=np.cumsum([0] + [order[key[gg]].values / mx[key[gg]] * 100
                                              for gg in gases[:k]], axis=0) if False else 0,
                height=0.5, color=cols[g], alpha=0.9, edgecolor=BG, lw=0.3,
                label=T(f"{g} (คิดเป็น % ของค่าสูงสุดของก๊าซนั้นทั้งประเทศ)"))
        break
    for k, g in enumerate(gases):
        vals = order[key[g]].values / mx[key[g]] * 100
        ax.scatter(vals, y + (k - 2) * 0.11, s=42, color=cols[g], edgecolor=INK, lw=0.4,
                   label=T(g), zorder=5 + k)
    ax.set_yticks(y); ax.set_yticklabels(order.region_th.values, color=INK, fontsize=10.5)
    ax.invert_yaxis(); ax.set_xlim(0, 108)
    ax.set_xlabel(T("ค่าเฉลี่ยความเข้มข้นคิดเป็นร้อยละของค่าสูงสุดในบรรดา 6 ภูมิภาค"))
    ax.set_title(T("กลิ่นของไทยต่างกันตามภูมิภาค — และก๊าซที่เด่นก็ไม่เหมือนกัน"),
                 color=INK, fontsize=12.5)
    ax.legend(facecolor="#111c2b", edgecolor="#33415e", labelcolor=INK, fontsize=9, ncol=5,
              loc="upper center", bbox_to_anchor=(0.5, -0.14))
    ax.grid(color="#26364f", lw=0.5, alpha=0.5, axis="x")
    ax.tick_params(colors=INK, labelsize=9.5)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color("#33415e")

    ax2.set_facecolor("#111c2b")
    ax2.scatter(order.smell_mean, [i for i in range(len(order))], s=90, color=ACCENT,
                edgecolor=INK, lw=0.5)
    for i, r in enumerate(order.itertuples()):
        ax2.text(r.smell_mean + 0.06, i, f"{r.dominant}", color=MUTED, fontsize=9.5, va="center")
    ax2.set_yticks(range(len(order))); ax2.set_yticklabels(order.region_th.values, color=INK, fontsize=10)
    ax2.invert_yaxis()
    ax2.set_xlabel(T("ดัชนีกลิ่นเฉลี่ยของภูมิภาค (z-score)"))
    ax2.set_title(T("ภูมิภาคที่กลิ่นแรงสุด (ตัวเลขข้างจุด = ก๊าซที่เด่น)"), color=INK, fontsize=12.5)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5, axis="x")
    ax2.tick_params(colors=INK, labelsize=9.5)
    for s in ("top", "right"): ax2.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax2.spines[s].set_color("#33415e")
    fig.suptitle(T("จมูกของแต่ละภูมิภาคไม่เหมือนกัน"), color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day19_regions.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ================================================== ภาพ 3 — กลิ่นกับฟ้า และกลิ่นกับแผนที่
def draw_scatter():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15.0, 6.0), dpi=150)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    ax.scatter(S.sqm.values, S.smell_index.values, s=54,
               c=[plt.get_cmap("inferno")(i) for i in np.linspace(0.15, 0.95, len(S))],
               edgecolor=INK, lw=0.4)
    ax.set_xlabel(T("คุณภาพท้องฟ้า SQM (mag/arcsec²) จากทะเบียน — ยิ่งมากยิ่งมืด"))
    ax.set_ylabel(T("ดัชนีกลิ่น (z-score)"))
    ax.set_title(T(f"ฟ้ามืดกับกลิ่น\nSpearman rho = {AGG['spearman_smell_vs_sqm']:.3f} "
                   f"(p = {AGG['p_smell_sqm']:.2g})"), color=INK, fontsize=12)
    ax.grid(color="#26364f", lw=0.5, alpha=0.5); ax.tick_params(colors=INK, labelsize=10)
    for s in ax.spines.values(): s.set_color("#33415e")

    ax2.set_facecolor("#111c2b")
    ax2.scatter(np.log10(np.clip(ways, 1, None)), S.smell_index.values, s=54, color=COOL,
                edgecolor=INK, lw=0.4)
    ax2.set_xlabel(T("log10 จำนวน way ในรัศมี 1 กม. รอบจุด (Day 16)"))
    ax2.set_ylabel(T("ดัชนีกลิ่น (z-score)"))
    ax2.set_title(T(f"แผนที่เยอะกว่า = กลิ่นแรงกว่าไหม\nSpearman rho = {AGG['spearman_smell_vs_ways']:.3f} "
                    f"(p = {AGG['p_smell_ways']:.2g})"), color=INK, fontsize=12)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5); ax2.tick_params(colors=INK, labelsize=10)
    for s in ax2.spines.values(): s.set_color("#33415e")
    fig.suptitle(T("สองคำถามที่ซีรีส์นี้ถามมาตลอด ในมุมของจมูก"), color=INK, fontsize=15, y=0.98)
    fig.text(0.5, 0.015, T("ค่าทั้งสองแกนเป็นแบบจำลอง CAMS/ERA5 ไม่ใช่การวัดหน้างาน "
                           "และเป็นหน้าต่างสี่วันเท่านั้น"), color=MUTED, fontsize=10, ha="center")
    p = os.path.join(OUT, "day19_smell_vs_dark.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_hero(), draw_regions(), draw_scatter()

## ตอน 4) artifact

In [ ]:
# ---------------------------------------------------------------- 4. artifact
keep = ["idx", "name", "province", "region_th", "lat", "lon", "sqm", "CH4", "SO2", "CO", "NO2",
        "PM2.5", "PM10", "O3", "dust", "NH3", "smell_index", "signature",
        "wind_from_dir", "wind_mean_speed", "z_CH4", "z_SO2", "z_CO", "z_NO2", "z_PM2.5"]
S[keep].round(4).to_csv(os.path.join(OUT, "day19_smell_sites.csv"), index=False)
by_region.round(4).to_csv(os.path.join(OUT, "day19_by_region.csv"), index=False)
STATS = dict(
    theme="How do you map things which your nose can sense? What does a smelly map look like?",
    theme_name="Senses: Smell",
    theme_source="30daymapchallenge.com หัวข้อ Day 19 (อ่านสดวันที่ 20 พ.ย.)",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    posted_late="เขียนคืนวันที่ 19 ต่อเช้าวันที่ 20 พ.ย. สำหรับโจทย์วันที่ 19 (ช้าไม่กี่ชั่วโมง)",
    window=dict(start="2026-11-16", end="2026-11-19", hours=96),
    sources=dict(air_quality="CAMS ผ่าน Open-Meteo Air Quality API (hourly, model grid ~0.1°)",
                 wind="ERA5 ผ่าน Open-Meteo Archive API (wind_direction_10m, wind_speed_10m)",
                 sites="ทะเบียนแหล่งท้องฟ้ามืด NARIT 83 จุด (ฟิลด์เดียวกับ Day 1/11/12/18)",
                 map_density="day16_sites_osm.csv (จำนวน way รัศมี 1 กม.)"),
    aggregates=AGG,
    by_region=[{k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v)
                for k, v in r.items()} for r in by_region.to_dict("records")],
    per_site=[{k: (None if (isinstance(v, float) and pd.isna(v)) else v) for k, v in r.items()}
              for r in S[keep].to_dict("records")],
    explanatory_numbers=dict(
        overpass_http_status=502,
        wind_from_to_offset_deg=180,
        wind_example_a_deg=350, wind_example_b_deg=10,
        wind_arithmetic_mean_wrong_deg=180,
        meaning="ตัวเลขประกอบคำอธิบายในโพสต์ (รหัส HTTP, มุมกลับทิศ from→to, "
                "ตัวอย่างการเฉลี่ยมุมที่ผิด) เก็บไว้เพื่อให้ guard_claims ตามที่มาได้"),
    unavailable=dict(
        ammonia="API คืน null ทั้ง 96 ชั่วโมงทุกจุดในไทย (CAMS global ไม่มีตัวแปร region นี้) "
                "ทั้งที่ NH3 คือกลิ่นหลักของนาข้าวและปศุสัตว์",
        dust="ค่า = 0 ทุกชั่วโมงทุกจุดในช่วงที่วัด",
        grid_note="จุดบางแห่งถูก map ไปยัง grid cell เดียวกัน (ค่าซ้ำกัน) เพราะ resolution ~0.1° "
                  "ใหญ่กว่าระยะระหว่างจุดหลายคู่"),
    figures=["day19_smell.png", "day19_regions.png", "day19_smell_vs_dark.png"],
    metric_definitions={
        "CH4 / SO2 / CO / NO2 / PM2.5": "ค่าเฉลี่ยรายชั่วโมงของความเข้มข้น (µg/m³) ที่จุดนั้น "
                                       "ในช่วง 16–19 พ.ย. 2569 จากแบบจำลอง CAMS",
        "z_GAS": f"(ค่าของจุด − ค่าเฉลี่ย 83 จุด) ÷ ส่วนเบี่ยงเบนมาตรฐานของ 83 จุด — "
                 "ทำให้ก๊าซห้าชนิดที่หน่วยต่างกันมาบวกกันได้",
        "smell_index": "ค่าเฉลี่ยของ z-score ห้าก๊าซ (CH4, SO2, CO, NO2, PM2.5) "
                       "เป็นตัวชี้วัดเชิงเปรียบเทียบระหว่างจุด ไม่ใช่หน่วยกลิ่น (ou/m³)",
        "signature": "ก๊าซที่มี z-score สูงสุดที่จุดนั้น = 'กลิ่นเด่น' ในเชิงแบบจำลอง",
        "wind_from_dir": "ทิศที่ลม *พัดมา* (องศาจากเหนือ ตามธรรมเนียมอุตุนิยมวิทยา) "
                         "คำนวณแบบเฉลี่ยเวกเตอร์ถ่วงด้วยความเร็วลม",
        "wind_mean_speed": "ความเร็วลมเฉลี่ย 10 ม. เหนือพื้นผิว (ม./วิ) ค่าเฉลี่ยเชิงสเกลเลอร์",
        "sqm": "คุณภาพท้องฟ้าจากทะเบียน (mag/arcsec²) — ฟิลด์เดียวกับที่ Day 1/18 ใช้",
        "ways": "จำนวน way ในรัศมี 1 กม. รอบจุดจาก Day 16 (ตัวชี้ว่าพื้นที่นั้นถูกเขียนแผนที่ไว้แค่ไหน)"},
    note="แผ่นนี้ตอบโจทย์ด้วยก๊าซที่พาโมเลกุลไปถึงจมูก ไม่ใช่ด้วยแบบสำรวจความรู้สึก "
         "และบอกชัดเจนว่าอะไรไม่มีข้อมูล (NH3) แทนที่จะเงียบ")
json.dump(STATS, open(os.path.join(OUT, "day19_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
log("เขียน day19_stats.json + day19_smell_sites.csv + day19_by_region.csv")

## ตอน 5) referee

In [ ]:
# ---------------------------------------------------------------- 5. referee
for f in ["day19_smell.png", "day19_regions.png", "day19_smell_vs_dark.png",
          "day19_smell_sites.csv", "day19_by_region.csv", "day19_stats.json"]:
    assert os.path.getsize(os.path.join(OUT, f)) > 0, f"ไฟล์ว่าง: {f}"
json.load(open(os.path.join(OUT, "day19_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day19-texts")
assert len(S) == 83 and S.smell_index.notna().all()
assert abs(float(np.mean(S.smell_index.values))) < 0.05, "z-score เฉลี่ยต้องใกล้ 0"
assert AGG["signature_counts"], "ต้องมี signature ทุกจุด"
assert "NH3" in AGG["no_data_variables"], "ต้องบันทึกว่า NH3 ไม่มีข้อมูล"
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทยในบรรทัดไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
log("สรุป: " + json.dumps(dict(smell_min=AGG["smell_min"], smell_median=AGG["smell_median"],
                               smell_max=AGG["smell_max"], signature=AGG["signature_counts"],
                               rho_sqm=AGG["spearman_smell_vs_sqm"], p_sqm=round(AGG["p_smell_sqm"], 4),
                               rho_ways=AGG["spearman_smell_vs_ways"], p_ways=round(AGG["p_smell_ways"], 4),
                               wind=AGG["wind_speed_mean"]), ensure_ascii=False))
print("DONE day19 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()